In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://www.rugbypass.com",
    "Referer": "https://www.rugbypass.com/",
})

In [66]:
x = session.get("https://www.rugbypass.com/live/france-vs-ireland/teams/?g=947104")

In [67]:
html = BeautifulSoup(x.text)

In [104]:
def player_id_from_url(url):
    return url.strip("/").split("/")[-1]

def count_goals(player, name):
    x = player.select_one(f".{name}")
    if x is None:
        return 0
    return len(x.find_all("div", recursive=False))
    
def process_team2(team):
    players = team.select("div.player")

    data = {}
    
    for player in players:
        name = player.select_one(".name").get_text(strip=True)
        number = int(player.select_one(".num").text.strip())

        conversions = count_goals(player, "conversion")
        penalties = count_goals(player, "penalty")
        drop_goals = count_goals(player, "drop_goal")

        player_identifier = player_id_from_url(player.select_one(".name").a["href"])
        
        data[name] = {
            "number": number, 
            "conversions": conversions, 
            "penaly_goals": penalties,
            "drop_goals": drop_goals,
            "player_identifier": player_identifier
        }
        
    return data

def process_teams2(soup):
    teams = soup.select("div.players")

    return {
        "home": process_team2(teams[0]),
        "away": process_team2(teams[1])
    } 

In [108]:
urls = pd.read_parquet("new_urls.parquet")

In [121]:
urls.index.unique()

Index([921086, 921087, 921088, 921089, 921090, 921091, 921092, 921093, 921094,
       921095,
       ...
       943602, 943595, 943627, 939016, 939008, 943623, 939027, 943604, 939026,
       943609],
      dtype='int64', name='match_id', length=2205)

In [130]:
matches = pd.read_parquet("matches.parquet")

In [133]:
urls = pd.read_parquet("new_urls.parquet")
matches = pd.read_parquet("matches.parquet")
perfs = pd.read_parquet("player_performances.parquet")
missing_match_ids = matches.index[
    ~matches.index.isin(perfs["match_id"])
]